# Sakura — HF async-training feature tour

Every knob `SakuraHFCallback` exposes, demonstrated on a tiny BERT fine-tune (SST-2 subset) so the notebook runs in about a minute on a CPU/MPS laptop.

| knob | what it does |
|---|---|
| `drain="lazy"` / `"strict"` | whether `on_epoch_end` blocks to reap the previous future |
| `cache_key=...` | keep the validator model architecture warm on the worker |
| `fp16_state_dict=True` | halve network bytes |
| `on_backpressure="skip"/"queue"/"block"` | behaviour when `AdaptiveCompute` reports saturation |
| `async_copy=True` (CUDA-only) | GPU→CPU snapshot on a dedicated `torch.cuda.Stream` |
| in-memory handle | automatic shortcut when the compute is standalone — skip `torch.save` entirely |

Reference `zakuro/PLAN.md` for the full measured numbers.

In [ ]:
import os
import time

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

import zakuro as zk
from sakura.huggingface import SakuraHFCallback

print("torch:", torch.__version__)
print("zakuro:", zk.__version__)

## Setup

Small model, small dataset — enough signal, minimal wall clock.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
TRAIN_SIZE = 200
VAL_SIZE = 400
MAX_LENGTH = 64
EPOCHS = 3
BATCH_SIZE = 32

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def _tokenize(batch):
    return tok(batch["sentence"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

raw = load_dataset("glue", "sst2")
train = raw["train"].shuffle(seed=42).select(range(TRAIN_SIZE)).map(_tokenize, batched=True)
val = raw["validation"].shuffle(seed=42).select(range(min(VAL_SIZE, len(raw["validation"])))).map(_tokenize, batched=True)
cols = ["input_ids", "attention_mask", "label"]
train.set_format("torch", columns=cols)
val.set_format("torch", columns=cols)

val_payload = (
    {
        "input_ids": torch.stack([val[i]["input_ids"] for i in range(len(val))]).clone(),
        "attention_mask": torch.stack([val[i]["attention_mask"] for i in range(len(val))]).clone(),
        "label": torch.stack([val[i]["label"] for i in range(len(val))]).clone(),
    },
    BATCH_SIZE,
)

def model_factory():
    return AutoModelForSequenceClassification.from_config(config)

def eval_fn(model, payload):
    import torch as _t
    data, bs = payload
    device = _t.device("cpu")
    model.to(device)
    correct = total = 0
    with _t.no_grad():
        for s in range(0, len(data["input_ids"]), bs):
            batch = {k: data[k][s:s+bs].to(device) for k in ("input_ids","attention_mask")}
            labels = data["label"][s:s+bs].to(device)
            out = model(**batch, labels=labels)
            correct += int((out.logits.argmax(-1) == labels).sum().item())
            total += int(labels.size(0))
    return {"val_acc": correct / max(total, 1)}

print(f"train={len(train)}  val={len(val)}  epochs={EPOCHS}")

In [ ]:
def training_args(outdir: str, eval_strategy: str) -> TrainingArguments:
    return TrainingArguments(
        output_dir=outdir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy=eval_strategy,
        save_strategy="no",
        logging_strategy="no",
        report_to=[],
        disable_tqdm=True,
        seed=42,
        dataloader_num_workers=0,
        fp16=torch.cuda.is_available(),
    )

def run_with_callback(callback: SakuraHFCallback, label: str) -> dict:
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    tr = Trainer(
        model=m,
        args=training_args(f"/tmp/hf-features/{label}", "no"),
        train_dataset=train,
        callbacks=[callback],
    )
    t0 = time.perf_counter()
    tr.train()
    wall = time.perf_counter() - t0
    return {"label": label, "wall": wall, "history": callback.history}

## 1. Baseline — default callback (lazy drain, cache on, in-memory handle auto-on for standalone)

Every default knob in effect. `val_compute=None` so Zakuro falls back to standalone and the in-memory handle path kicks in automatically.

In [ ]:
cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=None,         # standalone fallback
    verbose=False,
)
baseline = run_with_callback(cb, "defaults")
print(f"defaults: {baseline['wall']:.2f}s")
for h in cb.history:
    print(f"  epoch {int(h['epoch'])}: val_acc={h.get('val_acc'):.4f} took={h.get('elapsed_secs'):.2f}s")

## 2. `drain="strict"` — block at every epoch boundary

The lazy default (`drain="lazy"`) never blocks training at `on_epoch_end` — it only reaps futures that are already done and pushes the rest through the pool. `strict` reverts to the pre-v0.2 behaviour: each epoch waits for the previous eval to finish. Useful when the caller wants metrics in epoch-order, at the cost of wall time whenever eval > train.

In [ ]:
cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=None,
    drain="strict",
    verbose=False,
)
strict = run_with_callback(cb, "strict")
print(f"strict: {strict['wall']:.2f}s  (lazy was {baseline['wall']:.2f}s)")

## 3. `cache_key=None` — disable worker-side model cache

By default the validator architecture is cached in a module-level dict keyed by `cache_key` (default `"default"`). Setting `cache_key=None` disables the cache; every epoch re-runs `model_factory()` on the worker. Shows the cost of *not* caching.

In [ ]:
cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=None,
    cache_key=None,  # rebuild every epoch
    verbose=False,
)
no_cache = run_with_callback(cb, "no-cache")
print(f"no-cache: {no_cache['wall']:.2f}s  (cached was {baseline['wall']:.2f}s)")

## 4. `fp16_state_dict=True` — halve the bytes over the wire

Not visible in standalone mode — the state_dict never hits the wire — but the tensor dtype conversion still happens. Useful over a real network for fp32 models. For this demo we just confirm the path runs end-to-end.

In [ ]:
cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=None,
    fp16_state_dict=True,
    verbose=False,
)
fp16 = run_with_callback(cb, "fp16")
print(f"fp16: {fp16['wall']:.2f}s  (no-fp16 was {baseline['wall']:.2f}s)")

## 5. `on_backpressure` + `AdaptiveCompute`

Wrap the compute in `AdaptiveCompute` and flip the backpressure policy. With `on_backpressure="skip"` the callback drops an epoch's eval whenever the allocator reports saturation — the epoch's `history` row is `{"skipped": True}`. We artificially mark the allocator as backpressured to exercise the path cheaply.

In [ ]:
# An always-backpressured AdaptiveCompute for demonstration.
class AlwaysBackpressured(zk.AdaptiveCompute):
    def is_backpressured(self) -> bool:
        return True

adaptive = AlwaysBackpressured(workers=[zk.Compute()], backpressure_threshold=0.001)

cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=adaptive,
    on_backpressure="skip",
    verbose=False,
)
skipped_run = run_with_callback(cb, "bp-skip")
print(f"bp=skip: {skipped_run['wall']:.2f}s")
print("history:")
for h in cb.history:
    print(f"  {h}")
print(f"skipped rows: {sum(1 for h in cb.history if h.get('skipped'))}")

## 6. `async_copy` — CUDA-stream GPU→CPU snapshot

Only relevant when the training model lives on CUDA. The cell below detects the device and reports what would happen; on a CPU/MPS laptop the sync path is used automatically (measured as 0 delta).

In [ ]:
if torch.cuda.is_available():
    print("Running on CUDA — async_copy=True schedules the snapshot on a")
    print("dedicated torch.cuda.Stream. Measured on x399 4090: 176→75ms main-thread/epoch.")
    cb = SakuraHFCallback(
        model_factory=model_factory,
        eval_fn=eval_fn,
        eval_payload=val_payload,
        val_compute=None,
        async_copy=True,
        verbose=False,
    )
    async_run = run_with_callback(cb, "async-copy")
    print(f"async_copy=True: {async_run['wall']:.2f}s")
else:
    print("No CUDA here — async_copy falls back to the blocking .cpu() path.")
    print("See /opt/code/ZAK/zakuro/PLAN.md for the x399 4090 measurement:")
    print("  blocking .cpu()        : 176.1 ms/epoch main-thread")
    print("  async CUDA-stream copy :  74.8 ms/epoch main-thread  (−57.5 %)")

## 7. In-memory handle (auto)

When `val_compute` resolves to standalone (no URI, no host — which is the default when `val_compute=None`), the callback bypasses `torch.save`/`torch.load` entirely. The `state_dict` flows as a Python reference to the pool thread. Measured ~+24 % end-to-end wall on a 3-epoch distilbert standalone fine-tune (see `bert_demo/bench_in_memory_handle.py`).

The toggle is automatic — no API surface. To force the remote-style path for comparison, monkey-patch the detector.

In [ ]:
# Forcing the torch.save path even though we're in-process.
cb = SakuraHFCallback(
    model_factory=model_factory,
    eval_fn=eval_fn,
    eval_payload=val_payload,
    val_compute=None,
    verbose=False,
)
cb._is_in_process_target = lambda: False  # force torch.save path
torch_save = run_with_callback(cb, "torch-save")
print(f"torch.save path : {torch_save['wall']:.2f}s")
print(f"in-memory path  : {baseline['wall']:.2f}s")
delta = torch_save['wall'] - baseline['wall']
print(f"delta           : {delta:+.2f}s ({100*delta/torch_save['wall']:+.1f}%)")

## Summary table

In [ ]:
rows = [
    ("defaults (lazy drain, cache, in-memory handle)", baseline["wall"]),
    ("drain=strict", strict["wall"]),
    ("cache_key=None (re-instantiate each epoch)", no_cache["wall"]),
    ("fp16_state_dict=True", fp16["wall"]),
    ("on_backpressure=skip (AlwaysBackpressured)", skipped_run["wall"]),
    ("forced torch.save (no in-memory shortcut)", torch_save["wall"]),
]
width = 55
print(f"{'configuration':<{width}}  wall (s)")
print("-" * (width + 12))
for name, w in rows:
    print(f"{name:<{width}}  {w:>7.2f}")